In [1]:
import os
import time
import subprocess
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import pandas as pd

In [2]:
def compressImg(input_path, output_path):
    """
    Compress an img using bpg.
    """
    cmd = ['bpgenc', '-lossless', input_path, '-o', output_path]
    try:
        subprocess.run(cmd, check=True)
    except Exception as e:
        print(f'Error compressing {input_path}: {e}.')
        return False
    
    return True


def decompressImg(input_path, output_path):
    """
    DeCompress an bpg img.
    """
    cmd = ['bpgdec', '-o', output_path, input_path]
    try:
        subprocess.run(cmd, check=True)
    except Exception as e:
        print(f'Error decompressing {input_path}: {e}.')
        return False
    
    return True

In [3]:
def cal_compression_ratio(original_path, compressed_path):
    """
    Calculate compression rate based on file sizes.
    """
    original_size = os.path.getsize(original_path)
    compressed_size = os.path.getsize(compressed_path)

    with Image.open(original_path) as img:
        width, height = img.size
        npixels = width * height
    bpsp = (compressed_size * 8) / (npixels * 3)
    compression_ratio = compressed_size / original_size
    
    return compression_ratio, original_size, compressed_size, bpsp, width, height, npixels

In [4]:
def process_dataset(dataset_path, output_dir, mode="compress", cal_metric=True, verbose=False, keywords=None):
    """
    Process a dataset (Kodak or CLIC) using bpg, with optional compression rate calculation.
    """
    dataset_path = Path(dataset_path)
    output_dir   = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    imgs = list(dataset_path.rglob('*.png'))
    if keywords is not None:
        imgs = [img for img in imgs if keywords in img.name]
    
    if len(imgs) == 0:
        print(f'No Imgs found in {dataset_path}.')
        return
    
    rows = []
    for img in tqdm(imgs, desc=f'{mode.capitalize()}ing imgs'):
        output_path = output_dir / f'{img.stem}.bpg' if mode == 'compress' else output_dir / f'{img.stem}_recon.png'
        
        start = time.time()
        if mode == 'compress':
            success = compressImg(img, output_path)
        elif mode == 'decompress':
            success = decompressImg(img, output_path)
        elapsed = time.time() - start   
          
        if success and cal_metric and mode == 'compress':
            compression_ratio, original_size, compressed_size, bpsp, width, height, npixels = cal_compression_ratio(img, output_path)
            
            if verbose:
                print(f'{img.name}: compression ratio = {compression_ratio:.2%} (Original: {original_size} bytes, Compressed: {compressed_size} bytes), bpsp: {bpsp}.')
            rows.append([img.name, bpsp, compression_ratio, original_size, compressed_size, width, height, npixels, elapsed])
        
    return rows

In [6]:
""" 1. compress Touch and Go (png) """
dataset_path = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/compressed/bpg'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False, keywords='175211')
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])
# display(df)
# display(df.describe())

# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/bpg_TouchandGoDataset-v2.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/bpg_TouchandGoDataset-v2.csv').describe())


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed
count,1400.000000,1400.000000,1400.000000,1400.000000,1400.0,1400.0,1400.0,1400.000000
mean,1.292568,0.507475,288034.526429,148903.823571,640.0,480.0,307200.0,2.440584
std,0.398972,0.076424,44205.507025,45961.547668,0.0,0.0,0.0,0.142320
min,0.503576,0.301599,192348.000000,58012.000000,640.0,480.0,307200.0,1.929387
25%,1.030421,0.460349,253517.500000,118704.500000,640.0,480.0,307200.0,2.355305
50%,1.194310,0.490411,276483.000000,137584.500000,640.0,480.0,307200.0,2.426080
75%,1.374544,0.540726,310647.000000,158347.500000,640.0,480.0,307200.0,2.507050
max,3.246884,0.784402,481097.000000,374041.000000,640.0,480.0,307200.0,3.127393


In [7]:
""" 2. compress ObjectFolder (png) """
dataset_path = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/dataset-comp/test/image'
output_dir   = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/compressed/bpg'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])
# display(df)
# display(df.describe())

# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/bpg_ObjectFolder_1.0.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/bpg_ObjectFolder_1.0.csv').describe())


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.0,20000.0,20000.0,20000.000000
mean,3.725957,0.939213,28541.906000,26826.890550,160.0,120.0,19200.0,0.180371
std,0.385567,0.012496,2695.165442,2776.079775,0.0,0.0,0.0,0.005543
min,3.069028,0.890141,24187.000000,22097.000000,160.0,120.0,19200.0,0.161360
25%,3.442778,0.930477,26490.000000,24788.000000,160.0,120.0,19200.0,0.175824
50%,3.608542,0.938131,27698.000000,25981.500000,160.0,120.0,19200.0,0.180913
75%,3.954306,0.946667,30118.000000,28471.000000,160.0,120.0,19200.0,0.184156
max,5.875417,1.008126,41962.000000,42303.000000,160.0,120.0,19200.0,0.219329


In [7]:
""" 3. compress SSVTP (png) """
dataset_path = '/data/ssd/zhaoy/datasets/SSVTP/dataset-comp/test/image'
output_dir   = '/data/ssd/zhaoy/datasets/SSVTP/compressed/bpg'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])
# display(df)
# display(df.describe())

# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/bpg_SSVTP.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/bpg_SSVTP.csv').describe())


,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed
count,918.000000,918.000000,918.000000,918.000000,918.0,918.0,918.0,918.000000
mean,1.999821,0.895184,64301.678649,57594.847495,240.0,320.0,76800.0,2.081240
std,0.167368,0.008723,4853.413152,4820.204216,0.0,0.0,0.0,1.429680
min,1.674965,0.873810,53899.000000,48239.000000,240.0,320.0,76800.0,0.823254
25%,1.862057,0.889324,60208.000000,53627.250000,240.0,320.0,76800.0,1.326670
50%,1.974219,0.893452,63670.500000,56857.500000,240.0,320.0,76800.0,1.605368
75%,2.120443,0.899359,67915.000000,61068.750000,240.0,320.0,76800.0,2.181255
max,2.491875,0.934088,77399.000000,71766.000000,240.0,320.0,76800.0,11.147028


In [8]:
""" 4. compress ObjTac (png) """
dataset_path = '/data/ssd/zhaoy/datasets/ObjTac/dataset-comp/test/image'
output_dir   = '/data/ssd/zhaoy/datasets/ObjTac/compressed/bpg'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])
# display(df)
# display(df.describe())

# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/bpg_ObjTac.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/bpg_ObjTac.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed
count,51.000000,51.000000,51.000000,51.000000,51.0,51.000000,51.000000,51.000000
mean,0.512874,0.847037,11784.803922,10235.686275,60.0,903.352941,54201.176471,0.877714
std,0.313186,0.138250,7995.848535,7067.181482,0.0,391.917891,23515.073434,0.419137
min,0.026729,0.420538,818.000000,344.000000,60.0,331.000000,19860.000000,0.267720
25%,0.208211,0.814402,4868.500000,3429.500000,60.0,587.500000,35250.000000,0.541232
50%,0.555993,0.869107,10936.000000,10068.000000,60.0,813.000000,48780.000000,0.846012
75%,0.768229,0.931389,19075.500000,16807.000000,60.0,1142.000000,68520.000000,1.162200
max,1.135069,1.160804,28278.000000,24131.000000,60.0,1802.000000,108120.000000,1.921906


In [10]:
""" 5. compress YCB-Slide (png) """
dataset_path = '/data/ssd/zhaoy/datasets/YCB-Slide/dataset-comp/test/image'
output_dir   = '/data/ssd/zhaoy/datasets/YCB-Slide/compressed/bpg'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])
# display(df)
# display(df.describe())

# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/bpg_YCB-Slide.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/bpg_YCB-Slide.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed
count,10916.000000,10916.000000,10916.000000,10916.000000,10916.0,10916.0,10916.0,10916.000000
mean,1.921951,0.880160,62881.925247,55352.191096,240.0,320.0,76800.0,0.716665
std,0.090884,0.005613,2789.584906,2617.456944,0.0,0.0,0.0,0.020141
min,1.656354,0.854976,54243.000000,47703.000000,240.0,320.0,76800.0,0.661404
25%,1.857500,0.876575,60928.750000,53496.000000,240.0,320.0,76800.0,0.704346
50%,1.914375,0.880104,62656.000000,55134.000000,240.0,320.0,76800.0,0.713023
75%,1.974410,0.883787,64545.250000,56863.000000,240.0,320.0,76800.0,0.723675
max,2.283889,0.901743,73586.000000,65776.000000,240.0,320.0,76800.0,0.861317


In [10]:
dataset_path = '/data/ssd/zhaoy/datasets/Data_ICRA18/frames'
output_dir   = '/data/ssd/zhaoy/datasets/Data_ICRA18/compressed/bpg'

rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])
display(df)
display(df.describe())

df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/bpg_ActiveCloth.csv', index=False)

# display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/bpg_ActiveCloth.csv').describe())

Compressing imgs:   0%|          | 1/8455 [00:10<24:22:02, 10.38s/it]


KeyboardInterrupt: 